# KUKA Feature Selection Experiments for AE Anomaly Detection

Implemented feature-selection experiments:

1. all features;
2. near-constant feature removal;
3. near-constant + correlation-based redundancy pruning;
4. top-$m$ KS-statistic features;
5. top-$m$ AE reconstruction-separation features.

The split is produced by `dataset.kuka_splits.split_kuka_numpy`, the same splitting function used by the current `train_ae.py` and `train_aae.py`. Preprocessing is fitted only on training data. Supervised feature rankings use validation labels only. Test data are used only for final evaluation.

In [11]:
from pathlib import Path
import sys
import random

import numpy as np
import pandas as pd

import torch
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "dataset").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from dataset.kuka_splits import split_kuka_numpy
from dataset.kuka_dataset import KukaDataset
from processing.kuka_processing import KukaPreprocessor
from models.AE import AE
from metrics import compute_pr_auc

## 1. Configuration

Set `EPOCHS` low while debugging. For final experiments, use the same number of epochs used for your AE baseline.

In [12]:
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

SEED = 123
BATCH_SIZE = 1024
LATENT_DIM = 4
LR = 1e-3
EPOCHS = 50          # increase for final runs, e.g. 300 or 1500
TOP_K = 5
F_BETA = 2.0

CONSTANT_EPS = 1e-8
CORR_THRESHOLD = 0.98
M_VALUES = [10, 20, 40, 60]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cpu')

In [13]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

## 2. Load dataset split

In [14]:
splits = split_kuka_numpy(DATA_DIR, seed=SEED, binary_labels=True)

print("x_train:", splits.x_train.shape, "y_train:", splits.y_train.shape, np.unique(splits.y_train, return_counts=True))
print("x_val:  ", splits.x_val.shape,   "y_val:  ", splits.y_val.shape,   np.unique(splits.y_val, return_counts=True))
print("x_test: ", splits.x_test.shape,  "y_test: ", splits.y_test.shape,  np.unique(splits.y_test, return_counts=True))

n_features_raw = splits.x_train.shape[1]
print("Raw feature count:", n_features_raw)

x_train: (163654, 86) y_train: (163654,) (array([0]), array([163654]))
x_val:   (55837, 86) y_val:   (55837,) (array([0, 1]), array([35068, 20769]))
x_test:  (55839, 86) y_test:  (55839,) (array([0, 1]), array([35070, 20769]))
Raw feature count: 86


## 3. Load feature names robustly

`KukaNormal.npy` has no label column. `KukaSlow.npy` has the label as the last column. Therefore, the expected number of feature names is `splits.x_train.shape[1]`. If `KukaColumnNames.npy` contains one extra label name, the last name is dropped automatically.

In [15]:
def load_feature_names(data_dir: Path, n_features: int) -> np.ndarray:
    path = data_dir / "KukaColumnNames.npy"

    if not path.exists():
        print("KukaColumnNames.npy not found. Using generic feature names.")
        return np.array([f"feature_{i}" for i in range(n_features)], dtype=object)

    names = np.load(path, allow_pickle=True).astype(str)

    if len(names) == n_features:
        return names

    if len(names) == n_features + 1:
        print("Feature-name file has one extra column. Assuming last entry is label and dropping it.")
        return names[:-1]

    print(f"Unexpected number of feature names: {len(names)} for {n_features} features. Using generic feature names.")
    return np.array([f"feature_{i}" for i in range(n_features)], dtype=object)

feature_names = load_feature_names(DATA_DIR, n_features_raw)
assert len(feature_names) == n_features_raw

pd.DataFrame({"feature_index": np.arange(n_features_raw), "feature_name": feature_names}).head(10)

Feature-name file has one extra column. Assuming last entry is label and dropping it.


,feature_index,feature_name
0,0,action
1,1,machine_nameKuka Robot_apparent_power
2,2,machine_nameKuka Robot_current
3,3,machine_nameKuka Robot_frequency
4,4,machine_nameKuka Robot_phase_angle
5,5,machine_nameKuka Robot_power
6,6,machine_nameKuka Robot_power_factor
7,7,machine_nameKuka Robot_reactive_power
8,8,machine_nameKuka Robot_voltage
9,9,sensor_id1_AccX


## 4. Utility functions

These functions keep the notebook self-contained while reusing the project objects: `KukaPreprocessor`, `KukaDataset`, `AE`, and `compute_threshold`.

In [16]:
def make_dataloaders_for_features(selected_features, batch_size=BATCH_SIZE, normalization=True):
    """Build DataLoaders for a selected subset of original feature indices."""
    selected_features = np.asarray(selected_features, dtype=int)

    x_train_raw = splits.x_train[:, selected_features]
    x_val_raw = splits.x_val[:, selected_features]
    x_test_raw = splits.x_test[:, selected_features]

    preprocessor = KukaPreprocessor(
        normalization=normalization,
        remove_constant=False,
        corr_threshold=None,
        clip_percentiles=None,
    )

    x_train = preprocessor.fit_transform(x_train_raw)
    x_val = preprocessor.transform(x_val_raw)
    x_test = preprocessor.transform(x_test_raw)

    train_dataset = KukaDataset(x_train, splits.y_train)
    val_dataset = KukaDataset(x_val, splits.y_val)
    test_dataset = KukaDataset(x_test, splits.y_test)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, preprocessor


def ae_error_fn(pred, x, top_k=TOP_K):
    k = min(int(top_k), x.shape[1])
    return (pred - x).pow(2).topk(k, dim=1).values.mean(dim=1)


def train_ae_model(model, train_loader, device=DEVICE, lr=LR, epochs=EPOCHS):
    loss_fn = torch.nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    model.to(device)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for x, _ in train_loader:
            x = x.to(device)
            optimizer.zero_grad()
            recon = model(x)
            loss = loss_fn(recon, x)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)
        history.append(epoch_loss)

        if epoch == 0 or (epoch + 1) % max(1, epochs // 5) == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch + 1:4d}/{epochs}: loss={epoch_loss:.6f}")

    return history

@torch.no_grad()
def compute_reconstruction_error_scores(model, dataloader, recon_top_k):
    recon_scores = []
    labels = []

    model.eval()
    for x, y in dataloader:
        recon = model(x)
        
        if isinstance(recon, tuple):
            recon = recon[0]

        # reconstruction error score
        recon_score = (recon - x).pow(2).topk(recon_top_k, dim=1).values.mean(dim=1)

        recon_scores.append(recon_score.cpu())
        labels.append(y.cpu())

    recon_scores = torch.cat(recon_scores)
    labels = torch.cat(labels)

    return recon_scores, labels


def run_ae_experiment(experiment_name, selected_features, epochs=EPOCHS, seed=SEED):
    """Train and evaluate an AE on a specific feature subset."""
    selected_features = np.asarray(selected_features, dtype=int)

    if len(selected_features) == 0:
        raise ValueError(f"{experiment_name}: selected feature subset is empty")

    print("=" * 80)
    print(f"Experiment: {experiment_name}")
    print(f"Selected features: {len(selected_features)}")

    set_seed(seed)

    train_loader, val_loader, test_loader, preprocessor = make_dataloaders_for_features(selected_features)

    input_dim = len(selected_features)
    model = AE(input_dim, LATENT_DIM).to(DEVICE)

    history = train_ae_model(model, train_loader, epochs=epochs)

    
    scores, labels = compute_reconstruction_error_scores(model, val_loader, TOP_K)

    pr_auc = compute_pr_auc(
        scores=scores,
        labels=labels
    )

    row = {
        "experiment": experiment_name,
        "n_features": int(len(selected_features)),
        "pr_auc": pr_auc
    }

    print(pd.Series(row))

    return {
        "row": row,
        "model": model,
        "history": history,
        "selected_features": selected_features,
        "preprocessor": preprocessor,
        "labels": labels,
    }

## 5. Feature-selection methods

The methods below return indices relative to the original feature matrix.

In [18]:
def select_all_features(n_features: int) -> np.ndarray:
    return np.arange(n_features, dtype=int)


def select_nonconstant_features(x_train, eps=CONSTANT_EPS) -> np.ndarray:
    std = np.std(x_train, axis=0)
    return np.flatnonzero(std > eps)


def correlation_prune_features(x_train, candidate_features, threshold=CORR_THRESHOLD) -> np.ndarray:
    """Greedy correlation pruning in the order given by candidate_features."""
    candidate_features = np.asarray(candidate_features, dtype=int)
    x = x_train[:, candidate_features]

    if x.shape[1] <= 1:
        return candidate_features

    corr = np.corrcoef(x, rowvar=False)
    corr = np.nan_to_num(np.abs(corr), nan=0.0, posinf=0.0, neginf=0.0)

    keep_local = []
    for j in range(x.shape[1]):
        if not keep_local:
            keep_local.append(j)
            continue

        max_corr = np.max(corr[j, keep_local])
        if max_corr <= threshold:
            keep_local.append(j)

    return candidate_features[np.array(keep_local, dtype=int)]


def ks_statistic_1d(a, b):
    """Two-sample KS statistic D. Does not require scipy."""
    a = np.sort(np.asarray(a, dtype=float))
    b = np.sort(np.asarray(b, dtype=float))

    if len(a) == 0 or len(b) == 0:
        return np.nan

    values = np.concatenate([a, b])
    cdf_a = np.searchsorted(a, values, side="right") / len(a)
    cdf_b = np.searchsorted(b, values, side="right") / len(b)
    return float(np.max(np.abs(cdf_a - cdf_b)))


def rank_features_by_ks(x_val, y_val, candidate_features):
    candidate_features = np.asarray(candidate_features, dtype=int)
    y_val = np.asarray(y_val).astype(int)

    normal_mask = y_val == 0
    anomaly_mask = y_val == 1

    if normal_mask.sum() == 0 or anomaly_mask.sum() == 0:
        raise ValueError("Validation split must contain both normal and anomaly samples for KS ranking.")

    scores = []
    for f in candidate_features:
        scores.append(ks_statistic_1d(x_val[normal_mask, f], x_val[anomaly_mask, f]))

    scores = np.asarray(scores)
    order = np.argsort(scores)[::-1]

    return candidate_features[order], scores[order]


def rank_features_by_ae_reconstruction_separation(model, val_loader, candidate_features, eps=1e-8):
    """
    Rank selected input features by validation reconstruction-error separation:

        r_j = (mean_anomaly_error_j - mean_normal_error_j) / (std_normal_error_j + eps)

    candidate_features maps local model columns back to original feature indices.
    """
    model.eval()

    normal_errors = []
    anomaly_errors = []

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE)
            recon = model(x)
            feature_err = (recon - x).pow(2).cpu()

            y = y.cpu()
            if (y == 0).any():
                normal_errors.append(feature_err[y == 0])
            if (y == 1).any():
                anomaly_errors.append(feature_err[y == 1])

    if not normal_errors or not anomaly_errors:
        raise ValueError("Validation split must contain both normal and anomaly samples.")

    normal_errors = torch.cat(normal_errors, dim=0)
    anomaly_errors = torch.cat(anomaly_errors, dim=0)

    normal_mean = normal_errors.mean(dim=0)
    normal_std = normal_errors.std(dim=0)
    anomaly_mean = anomaly_errors.mean(dim=0)

    separation = ((anomaly_mean - normal_mean) / (normal_std + eps)).numpy()
    order = np.argsort(separation)[::-1]

    candidate_features = np.asarray(candidate_features, dtype=int)
    return candidate_features[order], separation[order]

## 6. Build feature subsets

In [19]:
all_features = select_all_features(n_features_raw)
nonconstant_features = select_nonconstant_features(splits.x_train, eps=CONSTANT_EPS)
nonconstant_corr_features = correlation_prune_features(
    splits.x_train,
    nonconstant_features,
    threshold=CORR_THRESHOLD,
)

print("All features:                 ", len(all_features))
print("After near-constant removal:  ", len(nonconstant_features))
print("After correlation pruning:    ", len(nonconstant_corr_features))

removed_constant = np.setdiff1d(all_features, nonconstant_features)
removed_corr = np.setdiff1d(nonconstant_features, nonconstant_corr_features)

print("Near-constant removed:")
display(pd.DataFrame({"feature_index": removed_constant, "feature_name": feature_names[removed_constant]}))

print("Correlation-pruned removed, first 20:")
display(pd.DataFrame({"feature_index": removed_corr[:20], "feature_name": feature_names[removed_corr[:20]]}))

All features:                  86
After near-constant removal:   82
After correlation pruning:     79
Near-constant removed:


,feature_index,feature_name
0,30,sensor_id2_temp
1,63,sensor_id5_temp
2,74,sensor_id6_temp
3,85,sensor_id7_temp


Correlation-pruned removed, first 20:


,feature_index,feature_name
0,5,machine_nameKuka Robot_power
1,6,machine_nameKuka Robot_power_factor
2,80,sensor_id7_GyroZ


In [20]:
# KS ranking uses validation labels only.
# Use nonconstant features as candidates to avoid selecting degenerate channels.

ks_ranked_features, ks_scores = rank_features_by_ks(
    splits.x_val,
    splits.y_val,
    candidate_features=nonconstant_features,
)

ks_table = pd.DataFrame({
    "rank": np.arange(1, len(ks_ranked_features) + 1),
    "feature_index": ks_ranked_features,
    "feature_name": feature_names[ks_ranked_features],
    "ks_score": ks_scores,
})

ks_table.head(20)

,rank,feature_index,feature_name,ks_score
0,1,37,sensor_id3_q1,0.444422
1,2,17,sensor_id1_q3,0.413103
2,3,15,sensor_id1_q1,0.411799
3,4,72,sensor_id6_q3,0.395636
4,5,71,sensor_id6_q2,0.359868
5,6,38,sensor_id3_q2,0.357046
6,7,8,machine_nameKuka Robot_voltage,0.335216
7,8,50,sensor_id4_q3,0.333629
8,9,4,machine_nameKuka Robot_phase_angle,0.308354
9,10,6,machine_nameKuka Robot_power_factor,0.308327


## 7. Run AE experiments up to KS-based feature selection

This section trains a new AE for each feature subset. It may take time depending on `EPOCHS`.

In [21]:
experiment_outputs = []
selected_feature_tables = {}

out_all = run_ae_experiment("all_features", all_features, epochs=EPOCHS, seed=SEED)
experiment_outputs.append(out_all)
selected_feature_tables["all_features"] = all_features

out_nc = run_ae_experiment("nonconstant", nonconstant_features, epochs=EPOCHS, seed=SEED)
experiment_outputs.append(out_nc)
selected_feature_tables["nonconstant"] = nonconstant_features

out_nc_corr = run_ae_experiment("nonconstant_corr", nonconstant_corr_features, epochs=EPOCHS, seed=SEED)
experiment_outputs.append(out_nc_corr)
selected_feature_tables["nonconstant_corr"] = nonconstant_corr_features

for m in M_VALUES:
    m_eff = min(m, len(ks_ranked_features))
    selected = ks_ranked_features[:m_eff]
    name = f"top_{m_eff}_ks"
    out = run_ae_experiment(name, selected, epochs=EPOCHS, seed=SEED)
    experiment_outputs.append(out)
    selected_feature_tables[name] = selected

Experiment: all_features
Selected features: 86
Epoch    1/50: loss=0.062107
Epoch   10/50: loss=0.018654
Epoch   20/50: loss=0.016381
Epoch   30/50: loss=0.014690
Epoch   40/50: loss=0.013837
Epoch   50/50: loss=0.013143
experiment    all_features
n_features              86
pr_auc            0.722222
dtype: object
Experiment: nonconstant
Selected features: 82
Epoch    1/50: loss=0.061263
Epoch   10/50: loss=0.017505
Epoch   20/50: loss=0.015494
Epoch   30/50: loss=0.012855
Epoch   40/50: loss=0.011900
Epoch   50/50: loss=0.011292
experiment    nonconstant
n_features             82
pr_auc            0.67768
dtype: object
Experiment: nonconstant_corr
Selected features: 79
Epoch    1/50: loss=0.065274
Epoch   10/50: loss=0.019475
Epoch   20/50: loss=0.016421
Epoch   30/50: loss=0.014584
Epoch   40/50: loss=0.013561
Epoch   50/50: loss=0.012818
experiment    nonconstant_corr
n_features                  79
pr_auc                0.711392
dtype: object
Experiment: top_10_ks
Selected features:

## 8. Top-$m$ AE reconstruction-separation features

This method first trains an AE on a candidate feature space, then ranks features using validation reconstruction-error separation:

$$
r_j = \frac{\mu_{anom,j} - \mu_{normal,j}}{\sigma_{normal,j} + \epsilon}.
$$

The ranking uses validation labels, not test labels. Here the candidate feature space is `nonconstant_corr_features`.

In [22]:
base_model_for_recon_ranking = out_nc_corr["model"]
base_candidate_features = nonconstant_corr_features

_, val_loader_for_base, _, _ = make_dataloaders_for_features(base_candidate_features)

ae_ranked_features, ae_sep_scores = rank_features_by_ae_reconstruction_separation(
    base_model_for_recon_ranking,
    val_loader_for_base,
    candidate_features=base_candidate_features,
)

ae_sep_table = pd.DataFrame({
    "rank": np.arange(1, len(ae_ranked_features) + 1),
    "feature_index": ae_ranked_features,
    "feature_name": feature_names[ae_ranked_features],
    "ae_reconstruction_separation": ae_sep_scores,
})

ae_sep_table.head(20)

,rank,feature_index,feature_name,ae_reconstruction_separation
0,1,50,sensor_id4_q3,12.993514
1,2,48,sensor_id4_q1,10.264585
2,3,49,sensor_id4_q2,6.606115
3,4,51,sensor_id4_q4,4.698593
4,5,60,sensor_id5_q2,2.014117
5,6,62,sensor_id5_q4,1.920820
6,7,38,sensor_id3_q2,1.320322
7,8,83,sensor_id7_q3,0.863053
8,9,82,sensor_id7_q2,0.700471
9,10,84,sensor_id7_q4,0.660610


In [23]:
for m in M_VALUES:
    m_eff = min(m, len(ae_ranked_features))
    selected = ae_ranked_features[:m_eff]
    name = f"top_{m_eff}_ae_recon_sep"
    out = run_ae_experiment(name, selected, epochs=EPOCHS, seed=SEED)
    experiment_outputs.append(out)
    selected_feature_tables[name] = selected

Experiment: top_10_ae_recon_sep
Selected features: 10
Epoch    1/50: loss=0.110249
Epoch   10/50: loss=0.011237
Epoch   20/50: loss=0.008962
Epoch   30/50: loss=0.008278
Epoch   40/50: loss=0.007318
Epoch   50/50: loss=0.007002
experiment    top_10_ae_recon_sep
n_features                     10
pr_auc                   0.724544
dtype: object
Experiment: top_20_ae_recon_sep
Selected features: 20
Epoch    1/50: loss=0.080117
Epoch   10/50: loss=0.019911
Epoch   20/50: loss=0.015579
Epoch   30/50: loss=0.013996
Epoch   40/50: loss=0.013073
Epoch   50/50: loss=0.011484
experiment    top_20_ae_recon_sep
n_features                     20
pr_auc                   0.776479
dtype: object
Experiment: top_40_ae_recon_sep
Selected features: 40
Epoch    1/50: loss=0.076402
Epoch   10/50: loss=0.021831
Epoch   20/50: loss=0.018337
Epoch   30/50: loss=0.016039
Epoch   40/50: loss=0.014747
Epoch   50/50: loss=0.013819
experiment    top_40_ae_recon_sep
n_features                     40
pr_auc          

## 9. Results summary

In [26]:
results_df = pd.DataFrame([out["row"] for out in experiment_outputs])

results_df.sort_values("pr_auc", ascending=False)

,experiment,n_features,pr_auc
9,top_40_ae_recon_sep,40,0.892075
4,top_20_ks,20,0.832197
5,top_40_ks,40,0.808333
3,top_10_ks,10,0.797754
8,top_20_ae_recon_sep,20,0.776479
6,top_60_ks,60,0.762130
10,top_60_ae_recon_sep,60,0.731540
7,top_10_ae_recon_sep,10,0.724544
0,all_features,86,0.722222
2,nonconstant_corr,79,0.711392


In [27]:
results_path = RESULTS_DIR / "feature_selection_results.csv"
ks_path = RESULTS_DIR / "feature_selection_ks_ranking.csv"
ae_sep_path = RESULTS_DIR / "feature_selection_ae_recon_sep_ranking.csv"
selected_path = RESULTS_DIR / "feature_selection_selected_features.csv"

results_df.to_csv(results_path, index=False)
ks_table.to_csv(ks_path, index=False)
ae_sep_table.to_csv(ae_sep_path, index=False)

selected_rows = []
for experiment, indices in selected_feature_tables.items():
    for rank, idx in enumerate(indices, start=1):
        selected_rows.append({
            "experiment": experiment,
            "rank": rank,
            "feature_index": int(idx),
            "feature_name": str(feature_names[idx]),
        })

selected_df = pd.DataFrame(selected_rows)
selected_df.to_csv(selected_path, index=False)

print("Saved results.")

Saved results.


## 11. Inspect selected features

Use these tables for the report and for deciding which feature-selection protocol to carry into the AE-vs-AAE comparison.

In [28]:
EXPERIMENT_TO_INSPECT = "top_10_ks"

if EXPERIMENT_TO_INSPECT in selected_feature_tables:
    idx = selected_feature_tables[EXPERIMENT_TO_INSPECT]
    display(pd.DataFrame({
        "rank": np.arange(1, len(idx) + 1),
        "feature_index": idx,
        "feature_name": feature_names[idx],
    }))
else:
    print("Available experiments:")
    print(list(selected_feature_tables.keys()))

,rank,feature_index,feature_name
0,1,37,sensor_id3_q1
1,2,17,sensor_id1_q3
2,3,15,sensor_id1_q1
3,4,72,sensor_id6_q3
4,5,71,sensor_id6_q2
5,6,38,sensor_id3_q2
6,7,8,machine_nameKuka Robot_voltage
7,8,50,sensor_id4_q3
8,9,4,machine_nameKuka Robot_phase_angle
9,10,6,machine_nameKuka Robot_power_factor
